In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - 분류: Multiverse Computing의 Qiskit Function
*[API 참조](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity)를 참조하세요*


<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
scikit-learn~=1.8.0
```
</AccordionItem>
</Accordion>
> **Note:** * Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, 및 On-Prem(IBM Quantum Platform API를 통한) Plan 사용자만 사용할 수 있는 실험적 기능입니다. 미리보기 릴리스 상태이며 변경될 수 있습니다.

## 개요
"Singularity Machine Learning - Classification" 함수를 사용하면, 양자 전문 지식 없이도 양자 하드웨어에서 실제 머신러닝 문제를 해결할 수 있습니다. 앙상블 방법 기반의 이 애플리케이션 함수는 하이브리드 분류기입니다. 초기 앙상블 훈련을 위해 부스팅, 배깅, 스태킹과 같은 고전적 방법을 활용합니다. 이후 변분 양자 고유값 솔버(VQE) 및 양자 근사 최적화 알고리즘(QAOA)과 같은 양자 알고리즘을 사용하여 훈련된 앙상블의 다양성, 일반화 능력, 전체 복잡성을 향상시킵니다.

다른 양자 머신러닝 솔루션과 달리, 이 함수는 대상 QPU의 qubit 수에 제한받지 않고 수백만 개의 예제와 특성을 가진 대규모 데이터셋을 처리할 수 있습니다. Qubit 수는 훈련할 수 있는 앙상블의 크기만 결정합니다. 또한 매우 유연하여, 금융, 의료, 사이버 보안을 포함한 광범위한 도메인의 분류 문제를 해결하는 데 사용할 수 있습니다.
고차원, 노이즈, 불균형 데이터셋과 관련된 고전적으로 어려운 문제에서 일관되게 높은 정확도를 달성합니다.
![작동 방식](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
다음과 같은 사용자를 위해 구축되었습니다:
1. 양자 머신러닝을 제품 및 서비스에 통합하여 기술 제공물을 향상시키려는 기업의 엔지니어와 데이터 과학자,
2. 양자 머신러닝 응용 프로그램을 탐색하고 분류 작업에 양자 컴퓨팅을 활용하려는 양자 연구소의 연구자, 그리고
3. 머신러닝과 같은 과정의 교육 기관에서 양자 컴퓨팅의 장점을 시연하려는 학생과 교사.

다음 예제는 `create`, `list`, `fit`, `predict`를 포함한 다양한 기능을 보여주며, 비선형 결정 경계로 인해 악명 높게 어려운 문제인 두 개의 인터리빙 반원으로 구성된 합성 문제에서의 사용법을 시연합니다.


## 함수 설명
이 Qiskit Function을 사용하면 Singularity의 양자 향상 앙상블 분류기를 사용하여 이진 분류 문제를 해결할 수 있습니다. 내부적으로, 레이블된 데이터셋에 대해 고전적으로 분류기 앙상블을 훈련한 다음, IBM&reg; QPU에서 양자 근사 최적화 알고리즘(QAOA)을 사용하여 최대 다양성과 일반화를 위해 최적화하는 하이브리드 접근 방식을 사용합니다. 사용자 친화적인 인터페이스를 통해 요구 사항에 따라 분류기를 구성하고, 선택한 데이터셋에 대해 훈련하며, 이전에 보지 못한 데이터셋에 대해 예측할 수 있습니다.

일반적인 분류 문제를 해결하려면:
1. 데이터셋을 전처리하고, 훈련 세트와 테스트 세트로 분할합니다. 선택적으로, 훈련 세트를 훈련 세트와 검증 세트로 추가 분할할 수 있습니다. [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html)을 사용하여 이를 수행할 수 있습니다.
2. 훈련 세트가 불균형한 경우, [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets)을 사용하여 클래스의 균형을 맞추기 위해 리샘플링할 수 있습니다.
3. 카탈로그의 `file_upload` 메서드를 사용하여 훈련, 검증, 테스트 세트를 함수의 스토리지에 개별적으로 업로드하고, 매번 관련 경로를 전달합니다.
4. 함수의 `create` 액션을 사용하여 양자 분류기를 초기화합니다. 학습기의 수와 유형, 정규화(람다 값), 레이어 수, 고전적 옵티마이저 유형, 양자 Backend 등을 포함한 최적화 옵션과 같은 하이퍼파라미터를 허용합니다.
5. 함수의 `fit` 액션을 사용하여 레이블된 훈련 세트(해당하는 경우 검증 세트 포함)를 전달하여 훈련 세트에서 양자 분류기를 훈련합니다.
6. 함수의 `predict` 액션을 사용하여 이전에 보지 못한 테스트 세트에 대해 예측합니다.
---
## 시작하기
[IBM Quantum Platform API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고, 다음과 같이 Qiskit Function을 선택합니다.

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## 예시
### 데이터셋 분류
이 예시에서는 "Singularity Machine Learning - Classification" 함수를 사용하여 두 개의 맞물린 반원 모양(moon-shaped)으로 이루어진 데이터셋을 분류합니다. 이 데이터셋은 합성 데이터이며, 2차원으로 구성되어 있고 이진 레이블로 표시되어 있습니다. 중심점 기반 클러스터링이나 선형 분류와 같은 알고리즘에 도전적인 과제가 되도록 설계되었습니다.
![Moons 데이터셋](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
이 과정을 통해 분류기를 생성하고, 훈련 데이터에 맞게 학습시키고, 테스트 데이터에 대한 예측에 활용하며, 완료 후 분류기를 삭제하는 방법을 알아볼 수 있습니다.
시작하기 전에 [scikit-learn](https://scikit-learn.org/)을 설치해야 합니다. 다음 명령어를 사용하여 설치하세요.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


다음 단계를 수행하세요.

1) [scikit-learn](https://scikit-learn.org/)의 [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) 함수를 사용하여 합성 데이터셋을 생성합니다.
2) 생성된 합성 데이터셋을 [공유 데이터 디렉터리](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html)에 업로드합니다.
3) [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create) 액션을 사용하여 양자 강화 분류기를 생성합니다.
4) [`list`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#list) 액션을 사용하여 분류기 목록을 확인합니다.
5) [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit) 액션을 사용하여 훈련 데이터로 분류기를 학습시킵니다.
6) [`predict`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#predict) 액션을 사용하여 학습된 분류기로 테스트 데이터를 예측합니다.
7) [`delete`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#delete) 액션을 사용하여 분류기를 삭제합니다.
8) 완료 후 정리 작업을 수행합니다.
**Step 1.** 필요한 모듈을 가져오고 합성 데이터셋을 생성한 다음, 훈련 및 테스트 데이터셋으로 분할합니다.

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 3.** Create a quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 4.** Train the quantum-enhanced classifier using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


**Step 3.** [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create) 액션을 사용하여 양자 강화 분류기를 생성합니다.

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


**Step 6.** Delete the quantum-enhanced classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

### `create_fit_predict` example

The following example demonstrates the [`create_fit_predict`](/docs/api/functions/multiverse-computing-singularity#create_fit_predict) action.

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Visit the [API reference](/docs/api/functions/multiverse-computing-singularity) for this Qiskit Function.
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117).

</Admonition>